In [1]:
import pandas as pd
from sklearn.preprocessing import StandardScaler, LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split
from keras.models import Model
from keras.layers import Dense, Input
from utils import *     
import oracledb
import numpy as np
import joblib

In [41]:
overtime_data = pd.read_excel(
    r'D:\surya.iyer\My Documents\projects\predictive analysis\PSIPL data\OT_DATA_PSIPL.xlsx',
    dtype={
        'EMP_MID': str,
        'SHIFT_NAME': str,
        
    },
    parse_dates=['Attendance Date','SHIFT_INTIME','SHIFT_OUTTIME','TOTAL_HRS'],
    date_format={
        'Attendance Date': '%m/%d/%Y',
        'SHIFT_INTIME': "%H:%M",
        'SHIFT_OUTTIME': "%H:%M",
        'TOTAL_HRS': "%H:%M"
    }
)

In [45]:
overtime_data['Shift Hours'] = ((overtime_data['SHIFT_OUTTIME'] - overtime_data['SHIFT_INTIME']).dt.total_seconds() / 3600).round()

In [44]:
overtime_data.loc[overtime_data['Shift Hours'] == -15, 'SHIFT_OUTTIME'] = overtime_data['SHIFT_OUTTIME'] + pd.Timedelta(days=1)

In [46]:
overtime_data = overtime_data[overtime_data['Shift Hours'] != 0]

In [47]:
overtime_data['Attendance Date'] = pd.to_datetime(overtime_data['Attendance Date'])
overtime_data['day'] = overtime_data['Attendance Date'].dt.day
overtime_data['month'] = overtime_data['Attendance Date'].dt.month
overtime_data['year'] = overtime_data['Attendance Date'].dt.year
overtime_data['day_of_week'] = overtime_data['Attendance Date'].dt.dayofweek

In [48]:
overtime_data['TOTAL_HRS'] = overtime_data['TOTAL_HRS'].dt.hour

In [49]:
window_size = 15
overtime_data['rolling_avg'] = overtime_data.groupby('EMP_MID')['TOTAL_HRS'].transform(lambda x: x.rolling(window_size).mean())
overtime_data['rolling_std'] = overtime_data.groupby('EMP_MID')['TOTAL_HRS'].transform(lambda x: x.rolling(window_size).std())
overtime_data['rolling_avg'] = overtime_data['rolling_avg'].fillna(0.0)
overtime_data['rolling_std'] = overtime_data['rolling_std'].fillna(0.0)

In [50]:
overtime_data[['rolling_avg','rolling_std']]

,rolling_avg,rolling_std
0,0.0,0.0
1,0.0,0.0
2,0.0,0.0
3,0.0,0.0
4,0.0,0.0
...,...,...
33354,0.0,0.0
33355,0.0,0.0
33356,0.0,0.0
33357,0.0,0.0


In [51]:
# Label encode employee number. Since label encoding requires inputs to be of uniform datatype, convert emp id to str
le = LabelEncoder()
overtime_data['emp_id_encoded'] = le.fit_transform(overtime_data['EMP_MID'].astype(str))

In [53]:
overtime_data['OT_HOURS'].isna().sum()

0

In [55]:
overtime_data[['EMP_MID','emp_id_encoded']]

,EMP_MID,emp_id_encoded
0,544052,1127
1,557489,1789
2,544424,1145
3,502242,91
4,544424,1145
...,...,...
33354,559546,1927
33355,559546,1927
33356,565969,2492
33357,559117,1896


In [56]:
# One-hot encode the 'Shift Name' column
shift_names = pd.get_dummies(overtime_data['SHIFT_NAME'], dtype=float)
overtime_data = pd.concat([overtime_data, shift_names], axis=1)

In [57]:
overtime_data.columns

Index(['COMP_ID', 'COMP_NAME', 'EMP_ID', 'EMP_MID', 'EMP_NAME',
       'DEPARTMENT_NAME', 'GRADE_NAME', 'Attendance Date', 'SHIFT_ID',
       'SHIFT_NAME', 'SHIFT_INTIME', 'SHIFT_OUTTIME', 'TOTAL_HRS', 'OT_HOURS',
       'Shift Hours', 'day', 'month', 'year', 'day_of_week', 'rolling_avg',
       'rolling_std', 'emp_id_encoded', '24 Hrs Shift', '9 HRS SHIFT', 'H',
       'U', 'V'],
      dtype='object')

In [58]:
# Scaling input features
scaler = MinMaxScaler()
# Scaling input features
X = overtime_data.loc[
    (overtime_data['TOTAL_HRS'].notna()) & (overtime_data['OT_HOURS'].notna()),
    ['OT_HOURS','TOTAL_HRS', 'day', 'month', 'year', 'day_of_week','24 Hrs Shift', '9 HRS SHIFT', 'H',
       'U', 'V', 'rolling_avg','rolling_std']
]

X = scaler.fit_transform(X)

In [59]:
X

array([[1.        , 0.        , 0.03448276, ..., 0.        , 0.        ,
        0.        ],
       [0.94240838, 0.        , 0.03448276, ..., 0.        , 0.        ,
        0.        ],
       [0.09424084, 0.73913043, 0.44827586, ..., 0.        , 0.        ,
        0.        ],
       ...,
       [0.        , 1.        , 0.75862069, ..., 0.        , 0.        ,
        0.        ],
       [0.        , 1.        , 0.72413793, ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.39130435, 0.13793103, ..., 0.        , 0.        ,
        0.        ]])

In [61]:
y = overtime_data.loc[(overtime_data['TOTAL_HRS'].notna()) & (overtime_data['OT_HOURS'].notna()),'emp_id_encoded'].reset_index(drop=True)

In [62]:
# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

input_dim = X.shape[1]
encoding_dim = 5
input_layer = Input(shape=(input_dim,))
encoder = Dense(encoding_dim, activation='relu')(input_layer)
decoder = Dense(input_dim, activation='sigmoid')(encoder)
autoencoder = Model(inputs=input_layer, outputs=decoder)

# Compile the Autoencoder model
autoencoder.compile(optimizer='adam', loss='mean_squared_error')

In [63]:
# Fit the Autoencoder model
autoencoder.fit(X_train, X_train, epochs=50, batch_size=10, verbose=0)

In [64]:
from datetime import datetime
today = datetime.now().strftime("%d-%m-%Y")
joblib.dump(value=autoencoder, filename = fr'D:\surya.iyer\My Documents\projects\predictive analysis\PSIPL models\burnout_prediction_model_{today}.pkl')

['D:\\surya.iyer\\My Documents\\projects\\predictive analysis\\PSIPL models\\burnout_prediction_model_16-06-2025.pkl']

In [83]:
# Use the Autoencoder to predict/reconstruct the input data
reconstructed_data = autoencoder.predict(X_test)
# reconstructed_data = pd.DataFrame(reconstructed_data).fillna(0).to_numpy()
# Calculate the reconstruction error (MSE) for each data point
reconstruction_error = np.mean((X_test - reconstructed_data) ** 2, axis=1)

# Identify anomalies based on the reconstruction error
threshold = np.percentile(reconstruction_error, 99)  # adjust the threshold as needed
anomaly_indices = np.where(reconstruction_error > threshold)[0]

# Get the number of anomalous employees
num_anomalous_employees = len(anomaly_indices)

# Get the anomalous employee IDs
anomalous_employee_ids = y_test.iloc[anomaly_indices]

202/202 ━━━━━━━━━━━━━━━━━━━━ 0s 875us/step


In [84]:
reconstructed_data

array([[4.9160097e-02, 3.4842753e-01, 4.8148668e-01, ..., 3.5917640e-04,
        1.8583812e-02, 1.4586546e-02],
       [4.1148186e-02, 9.6653062e-01, 8.5842884e-01, ..., 6.6539805e-09,
        9.7136754e-01, 2.9391622e-02],
       [4.9769722e-02, 9.6648490e-01, 6.0002762e-01, ..., 3.6143615e-09,
        9.6458185e-01, 2.6357809e-02],
       ...,
       [5.4278653e-02, 8.3418500e-01, 4.7997215e-01, ..., 3.3316063e-04,
        1.3000957e-02, 1.0903087e-02],
       [5.5291146e-02, 4.8484704e-01, 3.1585249e-01, ..., 2.4307567e-04,
        1.5817204e-02, 1.2735760e-02],
       [5.7007689e-02, 9.6456069e-01, 4.0005526e-01, ..., 3.4135152e-04,
        1.0587109e-02, 8.8118780e-03]], dtype=float32)

In [85]:
anomalous_employee_ids_original = le.inverse_transform(anomalous_employee_ids)

In [86]:
len(anomalous_employee_ids_original)

65

In [87]:
nw_view = overtime_data[overtime_data['EMP_MID'].isin(anomalous_employee_ids_original)]

In [88]:
nw_view.columns

Index(['COMP_ID', 'COMP_NAME', 'EMP_ID', 'EMP_MID', 'EMP_NAME',
       'DEPARTMENT_NAME', 'GRADE_NAME', 'Attendance Date', 'SHIFT_ID',
       'SHIFT_NAME', 'SHIFT_INTIME', 'SHIFT_OUTTIME', 'TOTAL_HRS', 'OT_HOURS',
       'Shift Hours', 'day', 'month', 'year', 'day_of_week', 'rolling_avg',
       'rolling_std', 'emp_id_encoded', '24 Hrs Shift', '9 HRS SHIFT', 'H',
       'U', 'V'],
      dtype='object')

In [89]:
burnout_employees_detail = pd.concat(
    {   
        "Employee ID": nw_view['EMP_MID'],
        "Employee Name": nw_view['EMP_NAME'],
        "Average OT Hours": nw_view['EMP_MID'].map(nw_view.groupby('EMP_MID')['TOTAL_HRS'].mean()).round(2),
        f"{window_size}-day rolling average of OT Hours": nw_view['EMP_MID'].map(nw_view.groupby('EMP_MID')['rolling_avg'].mean()).round(2)
    },
    axis=1
)

In [90]:
burnout_employees_detail.drop_duplicates().to_excel(r'D:\surya.iyer\My Documents\projects\predictive analysis\PSIPL model predictions\predicted_burnout_employees_data.xlsx', index=False)